# 03 - Multi-Trial Benchmarks (Champions & Classical Baselines)

This notebook consolidates the **Post-Evolution Benchmark Evaluation Pipeline** into a unified, reproducible workflow:
1. **Champion Extraction & Balance Check**: Selects top-performing candidate algorithms from the current SQLite database and exports `results/benchmark/champions.json`.
2. **Pre-Flight Diagnostic Audit**: Scans existing IOH traces in `results/ioh_traces/` and reports workload status.
3. **LLaMEA Champions Benchmark Execution**: Executes $N=20$ paired BBOB-instance trials per champion and reports clean objective gap ($\Delta y = f(x) - f^*$).
4. **Classical Baselines Benchmark Execution**: Executes the same $N=20$ instances for baseline optimizers (`CMA-ES`, `DE`, `PSO`).

The new protocol allows a **fixed 100,000 objective evaluations per trial** at every dimension, for champions and baselines. Synthesis targets 2D, 10D and 50D with the same allowance. The champion timeout remains 30 seconds.
**Storage warning:** configuration changes do not activate the separately prepared study. Complete the manual storage cutover before executing this notebook. Existing saved outputs describe the previous study, not new evaluations.
Results recorded under a different or unknown budget are marked for rerun, even with `force_rerun=false`; they are not resumed or counted as current completed coverage. Run both champion and baseline evaluation sections before refreshing analysis and figures.

The current schedule remains **instances 2–21**, with seed `42 + instance_id`; synthesis still uses instance 1. Multiple-instance synthesis is not implemented. Reuse requires matching budget, instance/seed identities and provenance, not just a completed run count. Restart the kernel before loading the updated settings.

In [3]:
# Setup paths and services
# Ensure project root src/ is in sys.path
import os
import sys
from pathlib import Path

cwd = Path('.').resolve()
root_dir = cwd.parent if cwd.name == 'notebooks' else cwd
src_dir = root_dir / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import json
import pandas as pd
from IPython.display import display, HTML

from shared.config import RESULTS_DIR
from bootstrap.benchmark import build_benchmark_workflow

workflow = build_benchmark_workflow(root_dir)
sqlite_repo = workflow.sqlite_repo
champ_service = workflow.champion_selection
benchmark_service = workflow.service
CHAMPIONS_PATH = RESULTS_DIR / 'benchmark' / 'champions.json'

# Evaluation settings & matrix conditions loaded directly from configs/benchmark.toml

print('✅ Benchmark evaluation environment initialized.')

✅ Benchmark evaluation environment initialized.


## 1. Champion Selection & Experiment Balance Check
Query the current synthesis database, inspect run balance, and export problem-specific champions to `results/benchmark/champions.json`.

In [4]:
summary, total_completed = sqlite_repo.get_experiment_balance()

# Refresh the exported champion snapshot from the database currently loaded in this project.
champions_dict, df_champions = champ_service.export_champions(CHAMPIONS_PATH)
print(f'✅ Exported {len(df_champions)} champions to {CHAMPIONS_PATH}')

if summary.empty:
    raise RuntimeError('No completed experiments found in database. Please run Notebook 02 first.')

print(f'== Total Completed Experiments: {total_completed} ==')
print(summary.to_string(index=False))

# The snapshot above is refreshed before evaluation, so it always matches the active DB.


✅ Exported 1920 champions to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/benchmark/champions.json
== Total Completed Experiments: 2160 ==
 problem_id  dim  noise_std     mode prompt_strategy                                    llm_name  completed_count
          1    2       0.00 explicit        baseline DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf                1
          1    2       0.00 explicit        baseline      qwen2.5-coder-14b-instruct-q4_k_m.gguf                1
          1    2       0.00 explicit        baseline      qwen2.5-coder-32b-instruct-q4_k_m.gguf                1
          1    2       0.00 explicit          guided DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf                1
          1    2       0.00 explicit          guided      qwen2.5-coder-14b-instruct-q4_k_m.gguf                1
          1    2       0.00 explicit          guided      qwen2.5-coder-32b-instruct-q4_k_m.gguf                1
          1    2       0.00 explicit        thinking DeepS

## 2. Pre-Flight Diagnostic Audit (Champions & Baselines Workload)
Audit pending vs. completed evaluation runs across all experimental conditions.

In [5]:
# 1. Load champions
with open(CHAMPIONS_PATH, "r", encoding="utf-8") as f:
    champions_raw = json.load(f)
champions_flat = benchmark_service.champion_selection.flatten_champions(champions_raw)

# ── Embedded HTML Dashboard Helper (Notebook-Local UI) ─────────────────────────
def render_html_dashboard(
    df_audit: pd.DataFrame,
    title: str = "Benchmark Evaluation Pre-Flight Audit",
    subtitle: str = "Real-time status of empirical evaluation runs",
    group_column: str = "model",
) -> str:
    """Render responsive HTML pre-flight dashboard for Jupyter display."""
    if df_audit.empty:
        return "<div>No audit data available.</div>"

    grp_col = (
        group_column
        if group_column in df_audit.columns
        else ("solver" if "solver" in df_audit.columns else df_audit.columns[0])
    )

    get_clean_model_label = benchmark_service.model_names.get_clean_model_label

    summary_rows = []
    for name, grp in df_audit.groupby(grp_col):
        clean_name = get_clean_model_label(str(name)) if grp_col == "model" else str(name).upper()
        total = len(grp)
        completed = len(grp[grp["status"] == "COMPLETED"])
        pending = len(grp[grp["status"] == "PENDING"])
        needs_rerun = len(grp[grp["status"] == "NEEDS_RERUN"])
        missing_code = len(grp[grp["status"] == "MISSING_CODE"])
        to_run_mask = grp["status"].isin(["PENDING", "NEEDS_RERUN"])
        if "is_filtered" in grp.columns:
            to_run_mask = to_run_mask & (~grp["is_filtered"])
        to_run = len(grp[to_run_mask])
        pct = (completed / total * 100) if total > 0 else 0.0
        summary_rows.append({
            "Group": clean_name,
            "Total Tasks": total,
            "Completed": completed,
            "Pending": pending,
            "Needs Rerun": needs_rerun,
            "Missing Code": missing_code,
            "Queue to Run": to_run,
            "Progress (%)": pct,
        })

    df_summary = pd.DataFrame(summary_rows)
    total_queue = int(df_summary["Queue to Run"].sum())
    total_completed = int(df_summary["Completed"].sum())
    total_tasks = int(df_summary["Total Tasks"].sum())
    overall_pct = (total_completed / total_tasks * 100) if total_tasks > 0 else 0.0

    html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 950px; margin: 15px 0;">
    <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 16px; border-bottom: 2px solid #E2E8F0; padding-bottom: 8px;">
        <div>
            <h2 style="margin: 0; color: #0F172A; font-size: 20px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
                🚀 {title}
            </h2>
            <p style="margin: 4px 0 0 0; color: #64748B; font-size: 13px;">{subtitle}</p>
        </div>
        <div style="background: #EEF2F6; padding: 6px 12px; border-radius: 20px; font-size: 12px; font-weight: 600; color: #334155;">
            {total_tasks} Target Conditions
        </div>
    </div>

    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 20px;">
        <div style="background: linear-gradient(135deg, #1E293B 0%, #0F172A 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #94A3B8;">Total Targets</div>
            <div style="font-size: 24px; font-weight: 700; color: #F8FAFC; margin-top: 4px;">{total_tasks}</div>
            <div style="font-size: 11px; color: #64748B; margin-top: 2px;">Across {len(df_summary)} Categories</div>
        </div>
        <div style="background: linear-gradient(135deg, #065F46 0%, #047857 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #A7F3D0;">Completed & Valid</div>
            <div style="font-size: 24px; font-weight: 700; color: #ECFDF5; margin-top: 4px;">{total_completed}</div>
            <div style="font-size: 11px; color: #D1FAE5; margin-top: 2px;">{overall_pct:.1f}% Overall Progress</div>
        </div>
        <div style="background: linear-gradient(135deg, #C2410C 0%, #9A3412 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #FED7AA;">Queue to Run</div>
            <div style="font-size: 24px; font-weight: 700; color: #FFF7ED; margin-top: 4px;">{total_queue}</div>
            <div style="font-size: 11px; color: #FFEDD5; margin-top: 2px;">Pending / Needs Rerun</div>
        </div>
        <div style="background: linear-gradient(135deg, #4338CA 0%, #3730A3 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #C7D2FE;">Overall Status</div>
            <div style="font-size: 24px; font-weight: 700; color: #EEF2FF; margin-top: 4px;">{overall_pct:.0f}%</div>
            <div style="font-size: 11px; color: #E0E7FF; margin-top: 2px;">Completion Rate</div>
        </div>
    </div>

    <div style="background: #F8FAFC; border: 1px solid #E2E8F0; border-radius: 10px; padding: 16px; margin-bottom: 8px;">
        <div style="font-size: 13px; font-weight: 700; color: #1E293B; margin-bottom: 12px; text-transform: uppercase; letter-spacing: 0.04em;">
            Completion Progress Breakdown
        </div>
"""
    for _, row in df_summary.iterrows():
        grp_name = row["Group"]
        tot = int(row["Total Tasks"])
        comp = int(row["Completed"])
        pend = int(row["Pending"])
        rerun = int(row["Needs Rerun"])
        q = int(row["Queue to Run"])
        pct = float(row["Progress (%)"])
        color = "#10B981" if pct > 75 else ("#3B82F6" if pct > 25 else "#F59E0B")
        rerun_pct = (rerun / tot * 100) if tot > 0 else 0

        html += f"""
        <div style="margin-bottom: 14px;">
            <div style="display: flex; justify-content: space-between; font-size: 13px; font-weight: 600; color: #334155; margin-bottom: 4px;">
                <span>⚙️ <strong style="color: #0F172A;">{grp_name}</strong> &nbsp;({comp}/{tot} Completed)</span>
                <span style="color: {color}; font-weight: 700;">{pct:.1f}%</span>
            </div>
            <div style="background: #E2E8F0; border-radius: 6px; height: 10px; overflow: hidden; display: flex;">
                <div style="background: #10B981; width: {pct}%; transition: width 0.3s;"></div>
                <div style="background: #EF4444; width: {rerun_pct}%;"></div>
            </div>
            <div style="display: flex; gap: 14px; font-size: 11px; color: #64748B; margin-top: 5px;">
                <span>✅ Completed: <strong style="color: #059669;">{comp}</strong></span>
                <span>⏳ Pending: <strong style="color: #D97706;">{pend}</strong></span>
                <span>⚠️ Needs Rerun: <strong style="color: #DC2626;">{rerun}</strong></span>
                <span>🎯 Queue: <strong style="color: #2563EB;">{q}</strong></span>
            </div>
        </div>
"""
    html += """
    </div>
</div>
"""
    return html

# 2. Audit champions workload (general overview by model)
df_champ_audit = benchmark_service.workload.audit_champions_workload()
print(f"🏆 Champions Workload: {len(df_champ_audit)} conditions ({df_champ_audit["status"].value_counts().to_dict()})")
display(HTML(render_html_dashboard(df_champ_audit, title="🏆 LLaMEA Champions Evaluation Audit", subtitle="Empirical benchmark evaluation status across LLM-evolved algorithms", group_column="model")))

# 3. Discover unique conditions for classical baselines (general overview by solver)
UNIQUE_CONFIGS = sorted(list({(c["dim"], c["noise_std"], c["problem_id"]) for c in champions_flat.values()}))
df_base_audit = benchmark_service.workload.audit_baselines_workload()
print(f"⚙️ Baselines Workload: {len(df_base_audit)} conditions ({df_base_audit["status"].value_counts().to_dict()})")
display(HTML(render_html_dashboard(df_base_audit, title="⚙️ Classical Baselines Evaluation Audit", subtitle="Empirical benchmark evaluation status across CMA-ES, DE, and PSO", group_column="solver")))

🏆 Champions Workload: 1920 conditions ({'PENDING': 1908, 'COMPLETED': 12})


⚙️ Baselines Workload: 240 conditions ({'PENDING': 240})


## 3. Execute LLaMEA Champions Benchmark (N=20 Independent Runs)
Evaluate evolved algorithms across target BBOB functions with full IOHprofiler `.dat` and `.json` logging.

In [6]:
results_champ_df = benchmark_service.run_champions()
print(f'✅ LLaMEA Champions evaluation complete: {len(results_champ_df)} conditions processed.')



🚀 STARTING BENCHMARK EVALUATIONS FOR CHAMPIONS
   1920 target conditions to evaluate

[1/1920] 🏆 Champion: DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf
  Mode:        explicit   Strategy:    baseline       Dim:         2D     Noise level: σ=0       Problem:     f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Δy Error: 0.000). Skipping.

[2/1920] 🏆 Champion: DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf
  Mode:        explicit   Strategy:    baseline       Dim:         2D     Noise level: σ=0.05    Problem:     f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Δy Error: 0.000). Skipping.

[3/1920] 🏆 Champion: DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf
  Mode:        explicit   Strategy:    baseline       Dim:         2D     Noise level: σ=0.1     Problem:     f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Δy Error: 0.000). Skipping.

[4/1920] 🏆 Champion: DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf
  Mode:        explicit   Strategy:    baseline       Dim:         2D     Noise 

<string>:29: UserWarning: [BUDGET OVERRUN] Problem evaluation called after budget exhausted (30000 >= 30000). Returning last observed fitness.
<string>:39: UserWarning: [BUDGET OVERRUN] Problem evaluation called after budget exhausted (30000 >= 30000). Returning last observed fitness.


  • ⚡ Trial 20/20 | Instance: 21 | Seed: 63 | f-opt: 115.170 | Best f: 115.195 | Δy Error: 0.025 | Evals: 30,000 | Time: 0.152s
  ✨ [COMPLETED] 20 runs finished | Median Δy Error: 5.814

[311/1920] 🏆 Champion: DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf
  Mode:        implicit   Strategy:    thinking       Dim:         3D     Noise level: σ=0.1     Problem:     f11 (Discus (f11))
  🔄 [RESUMING] Found 19/20 completed runs. Continuing from Trial 20...
  • ⚡ Trial 20/20 | Instance: 21 | Seed: 63 | f-opt: 115.170 | Best f: 116.611 | Δy Error: 1.441 | Evals: 30,000 | Time: 0.489s
  ✨ [COMPLETED] 20 runs finished | Median Δy Error: 2.537

[312/1920] 🏆 Champion: DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf
  Mode:        implicit   Strategy:    thinking       Dim:         3D     Noise level: σ=0.2     Problem:     f11 (Discus (f11))
  🔄 [RESUMING] Found 19/20 completed runs. Continuing from Trial 20...
  • ⚡ Trial 20/20 | Instance: 21 | Seed: 63 | f-opt: 115.170 | Best f: 115.371 | Δy Error: 0

## 4. Execute Classical Baselines Benchmark (CMA-ES, DE, PSO)
Run classical baseline optimizers across identical problem conditions for rigorous comparative benchmarking.

In [7]:
results_base_df = benchmark_service.run_baselines()
print(f'✅ Classical Baselines evaluation complete: {len(results_base_df)} conditions processed.')



🚀 STARTING BENCHMARK EVALUATIONS FOR BASELINES
   240 target conditions to evaluate

[1/240] ⚙️ Baseline: CMA-ES
  Mode:        —          Strategy:    —              Dim:         2D     Noise level: σ=0       Problem:     f1 (Sphere (f1))
  🔄 [RESUMING] Found 19/20 completed runs. Continuing from Trial 20...
  • ⚡ Trial 20/20 | Instance: 21 | Seed: 63 | f-opt: -133.640 | Best f: -133.640 | Δy Error: 0.000 | Evals: 426 | Time: 0.019s
  ✨ [COMPLETED] 20 runs finished | Median Δy Error: 0.000

[2/240] ⚙️ Baseline: CMA-ES
  Mode:        —          Strategy:    —              Dim:         2D     Noise level: σ=0       Problem:     f8 (Rosenbrock (f8))
  🔄 [RESUMING] Found 19/20 completed runs. Continuing from Trial 20...
  • ⚡ Trial 20/20 | Instance: 21 | Seed: 63 | f-opt: -44.640 | Best f: -44.640 | Δy Error: 0.000 | Evals: 552 | Time: 0.022s
  ✨ [COMPLETED] 20 runs finished | Median Δy Error: 0.000

[3/240] ⚙️ Baseline: CMA-ES
  Mode:        —          Strategy:    —              Dim:  

## Separate robustness and generalization campaigns

Noise robustness freezes a clean champion and varies noise on its original function.
Cross-function generalization freezes a clean, explicit, baseline-strategy champion
and changes only the target function at the original dimension. These are secondary
analyses, not part of the native champion ranking. Trials use instances 2–21,
excluding synthesis instance 1. Previously inspected instances 2–20 are not a new untouched test set.
Enable cross-function evaluation in `configs/benchmark.toml` before running it.
The switches below deliberately default to audit-only: no extra campaign runs on
notebook execution unless explicitly requested.


In [8]:
RUN_NOISE_ROBUSTNESS = False
RUN_CROSS_FUNCTION = False

noise_audit = benchmark_service.workload.audit_noise_robustness_workload()
transfer_audit = benchmark_service.workload.audit_cross_function_workload()
print(f"Noise robustness: {len(noise_audit)} conditions")
print(f"Cross-function generalization: {len(transfer_audit)} conditions (opt-in)")
display(noise_audit)
display(transfer_audit)

if RUN_NOISE_ROBUSTNESS:
    noise_results = benchmark_service.run_noise_robustness()
if RUN_CROSS_FUNCTION:
    if not benchmark_service.config.cross_function_enabled:
        raise ValueError("Enable benchmarking.cross_function_enabled in benchmark.toml first.")
    transfer_results = benchmark_service.run_cross_function_benchmarks()

# Always export the current readiness matrix, without treating missing trials as failures.
from shared.config import RESULTS_DIR
transfer_report_dir = RESULTS_DIR / "reports"
transfer_report_dir.mkdir(parents=True, exist_ok=True)
benchmark_service.workload.audit_noise_robustness_workload().to_csv(
    transfer_report_dir / "noise_robustness_readiness_v1.csv", index=False
)
benchmark_service.workload.audit_cross_function_workload().to_csv(
    transfer_report_dir / "cross_function_readiness_v1.csv", index=False
)


Noise robustness: 720 conditions
Cross-function generalization: 0 conditions (opt-in)


,key,raw_key,solver_type,solver,display_name,model,strategy,problem_id,source_problem_id,source_noise_std,...,dim,noise_std,mode,target_runs,runs_found,recorded_trials,reason,status,median_error,is_filtered
0,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf/f1...,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf/f1...,noise_robustness,deepseek_coder_v2_lite_instruct_q4_k_m_baseline,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf (n...,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,baseline,1,1,0.0,...,2,0.05,explicit,20,0,0,missing_provenance,PENDING,None,False
1,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf/f1...,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf/f1...,noise_robustness,deepseek_coder_v2_lite_instruct_q4_k_m_baseline,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf (n...,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,baseline,1,1,0.0,...,2,0.10,explicit,20,0,0,missing_provenance,PENDING,None,False
2,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf/f1...,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf/f1...,noise_robustness,deepseek_coder_v2_lite_instruct_q4_k_m_baseline,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf (n...,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,baseline,1,1,0.0,...,2,0.20,explicit,20,0,0,missing_provenance,PENDING,None,False
3,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf/f1...,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf/f1...,noise_robustness,deepseek_coder_v2_lite_instruct_q4_k_m_guided,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf (n...,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,guided,1,1,0.0,...,2,0.05,explicit,20,0,0,missing_provenance,PENDING,None,False
4,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf/f1...,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf/f1...,noise_robustness,deepseek_coder_v2_lite_instruct_q4_k_m_guided,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf (n...,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,guided,1,1,0.0,...,2,0.10,explicit,20,0,0,missing_provenance,PENDING,None,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
715,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_10D...,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_10D...,noise_robustness,qwen_32b_thinking,qwen2.5-coder-32b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-32b-instruct-q4_k_m.gguf,thinking,21,21,0.0,...,10,0.10,explicit,20,0,0,missing_provenance,PENDING,None,False
716,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_10D...,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_10D...,noise_robustness,qwen_32b_thinking,qwen2.5-coder-32b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-32b-instruct-q4_k_m.gguf,thinking,21,21,0.0,...,10,0.20,explicit,20,0,0,missing_provenance,PENDING,None,False
717,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_10D...,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_10D...,noise_robustness,qwen_32b_vectorization,qwen2.5-coder-32b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-32b-instruct-q4_k_m.gguf,vectorization,21,21,0.0,...,10,0.05,explicit,20,0,0,missing_provenance,PENDING,None,False
718,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_10D...,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_10D...,noise_robustness,qwen_32b_vectorization,qwen2.5-coder-32b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-32b-instruct-q4_k_m.gguf,vectorization,21,21,0.0,...,10,0.10,explicit,20,0,0,missing_provenance,PENDING,None,False


""


## 5. Post-evaluation outcome summary and synthesis review

Read-only summary of current native champions and baselines. This does not run experiments, change results or write reports. Historical unexecuted tails are excluded. Valid trials exclude execution failures; execution and failure rates use executed trials as their denominator. Target success rate counts trials reaching the primary error target, including failures in its denominator. The overall median includes failures, while the median finite error describes only successful executions. Missing/stale and incomplete conditions require benchmark completion before synthesis decisions. `all_failed` and `mostly_failed` (>50% execution failures) are review candidates, not automatic reruns. Median error above the configured secondary target is a performance flag, not proof of a bug. The full tables remain available as `evaluation_summary`, `failure_review` and `large_error_review`.


In [9]:
evaluation_summary = benchmark_service.summarize_results()
if evaluation_summary.empty:
    print("No current benchmark conditions to summarize.")
else:
    print(f"Summary of {len(evaluation_summary)} conditions; primary target={benchmark_service.config.reliability.primary_target:g}, secondary target={benchmark_service.config.reliability.secondary_target:g}")
    display(pd.crosstab(evaluation_summary["Model"], evaluation_summary["Category"]))
    review_columns = ["Model", "Experiment ID", "Algorithm", "Mode", "Strategy", "Problem ID", "Dim", "Noise", "Executed trials", "Valid trials", "Expected trials", "Failed trials", "Execution rate", "Failure rate", "Solved trials", "Primary target rate", "Median error", "Median finite error", "Best baseline median error", "Category", "Suggested action"]
    failure_review = evaluation_summary[evaluation_summary["Category"].isin(["all_failed", "mostly_failed", "some_failed", "incomplete", "missing_or_stale"])][review_columns]
    large_error_review = evaluation_summary[evaluation_summary["Category"] == "large_error"][review_columns]
    formats = {"Execution rate": "{:.1%}", "Failure rate": "{:.1%}", "Primary target rate": "{:.1%}", "Noise": "{:g}", "Median error": "{:.3f}", "Median finite error": "{:.3f}", "Best baseline median error": "{:.3f}"}
    print(f"Failure/readiness review: {len(failure_review)} conditions (first 100 shown).")
    display(failure_review.head(100).style.format(formats, na_rep="—"))
    print(f"Large finite-error review: {len(large_error_review)} conditions (first 50 shown; compare with baselines before rerunning).")
    display(large_error_review.head(50).style.format(formats, na_rep="—"))
    print("Suggested experiment IDs to REVIEW for fresh synthesis, grouped by owning LLM:")
    for model, group in evaluation_summary[evaluation_summary["Rerun candidate"]].groupby("Model", sort=True):
        print(f"{model}: {sorted(group["Experiment ID"].dropna().astype(int).unique().tolist())}")
    print("No reruns launched. Review diagnostics first; explicitly select IDs in Notebook 02/configuration.")


Summary of 2160 conditions; primary target=1e-08, secondary target=0.01


Category,finite_results,large_error,mostly_failed,some_failed
Model,,,,
DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,238,389,4,9
cmaes,31,49,0,0
de,18,62,0,0
pso,32,48,0,0
qwen2.5-coder-14b-instruct-q4_k_m.gguf,216,402,6,16
qwen2.5-coder-32b-instruct-q4_k_m.gguf,255,370,2,13


Failure/readiness review: 50 conditions (first 100 shown).


,Model,Experiment ID,Algorithm,Mode,Strategy,Problem ID,Dim,Noise,Executed trials,Valid trials,Expected trials,Failed trials,Execution rate,Failure rate,Solved trials,Primary target rate,Median error,Median finite error,Best baseline median error,Category,Suggested action
2,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2066,SRS,explicit,guided,1,2,0,20,19,20,1,95.0%,5.0%,0,0.0%,0.000,0.000,0.000,some_failed,Inspect failed-trial diagnostics
71,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2133,AdaptiveEvolutionaryAlgorithm,implicit,vectorization,1,5,0,20,13,20,7,65.0%,35.0%,0,0.0%,0.001,0.000,0.000,some_failed,Inspect failed-trial diagnostics
73,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2139,ImprovedSES,implicit,baseline,1,5,0.05,20,8,20,12,40.0%,60.0%,0,0.0%,inf,0.009,0.000,mostly_failed,Inspect; consider fresh synthesis
258,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2322,AdaptiveDifferentialEvolution,explicit,guided,11,2,0,20,9,20,11,45.0%,55.0%,8,40.0%,inf,0.000,0.000,mostly_failed,Inspect; consider fresh synthesis
282,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2720,MyOptimizer,explicit,guided,11,2,0.2,20,10,20,10,50.0%,50.0%,0,0.0%,inf,4.421,0.006,some_failed,Inspect failed-trial diagnostics
299,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2366,DifferentialEvolution,implicit,guided,11,3,0.05,20,10,20,10,50.0%,50.0%,0,0.0%,inf,13.650,0.000,some_failed,Inspect failed-trial diagnostics
300,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2360,StochasticRandomSearch,explicit,thinking,11,3,0.05,20,7,20,13,35.0%,65.0%,0,0.0%,inf,19710571.276,0.000,mostly_failed,Inspect; consider fresh synthesis
337,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2403,CMAESOptimizer,implicit,baseline,11,5,0.1,20,15,20,5,75.0%,25.0%,0,0.0%,12.671,4.937,0.085,some_failed,Inspect failed-trial diagnostics
452,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2512,AdaptiveEvolutionaryStrategyRefined,explicit,thinking,15,5,0,20,18,20,2,90.0%,10.0%,0,0.0%,44.660,44.446,2.026,some_failed,Inspect failed-trial diagnostics
502,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2561,AdaptiveDifferentialEvolutionRefined,explicit,vectorization,15,10,0.1,20,12,20,8,60.0%,40.0%,0,0.0%,515.247,419.078,17.605,some_failed,Inspect failed-trial diagnostics


Large finite-error review: 1320 conditions (first 50 shown; compare with baselines before rerunning).


,Model,Experiment ID,Algorithm,Mode,Strategy,Problem ID,Dim,Noise,Executed trials,Valid trials,Expected trials,Failed trials,Execution rate,Failure rate,Solved trials,Primary target rate,Median error,Median finite error,Best baseline median error,Category,Suggested action
5,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2068,AdaptiveRandomSearch,implicit,thinking,1,2,0,20,20,20,0,100.0%,0.0%,8,40.0%,0.054,0.054,0.000,large_error,Compare with baselines; not an automatic rerun
13,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2076,AdaptiveDE,implicit,thinking,1,2,0.05,20,20,20,0,100.0%,0.0%,8,40.0%,0.042,0.042,0.000,large_error,Compare with baselines; not an automatic rerun
14,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2073,DifferentialEvolution,explicit,vectorization,1,2,0.05,20,20,20,0,100.0%,0.0%,0,0.0%,0.019,0.019,0.000,large_error,Compare with baselines; not an automatic rerun
22,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2081,StochasticEA,explicit,vectorization,1,2,0.1,20,20,20,0,100.0%,0.0%,0,0.0%,0.220,0.220,0.000,large_error,Compare with baselines; not an automatic rerun
29,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2092,ImprovedRandomSearch,implicit,thinking,1,2,0.2,20,20,20,0,100.0%,0.0%,0,0.0%,4.165,4.165,0.000,large_error,Compare with baselines; not an automatic rerun
30,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2089,ParticleSwarmOptimization,explicit,vectorization,1,2,0.2,20,20,20,0,100.0%,0.0%,0,0.0%,0.404,0.404,0.000,large_error,Compare with baselines; not an automatic rerun
31,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2093,AlgorithmName,implicit,vectorization,1,2,0.2,20,20,20,0,100.0%,0.0%,0,0.0%,3.672,3.672,0.000,large_error,Compare with baselines; not an automatic rerun
37,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2100,SimpleEvolutionaryAlgorithm,implicit,thinking,1,3,0,20,20,20,0,100.0%,0.0%,0,0.0%,0.026,0.026,0.000,large_error,Compare with baselines; not an automatic rerun
51,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2118,MyOptimizer,implicit,guided,1,3,0.1,20,20,20,0,100.0%,0.0%,0,0.0%,134.779,134.779,0.000,large_error,Compare with baselines; not an automatic rerun
52,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,2112,AdaptiveStochasticHillClimbing,explicit,thinking,1,3,0.1,20,20,20,0,100.0%,0.0%,7,35.0%,7.729,7.729,0.000,large_error,Compare with baselines; not an automatic rerun


Suggested experiment IDs to REVIEW for fresh synthesis, grouped by owning LLM:
DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf: [2139, 2322, 2360, 2657]
qwen2.5-coder-14b-instruct-q4_k_m.gguf: [49, 177, 268, 327, 331, 416]
qwen2.5-coder-32b-instruct-q4_k_m.gguf: [1028, 1132]
No reruns launched. Review diagnostics first; explicitly select IDs in Notebook 02/configuration.
